# Face Recognition with OpenCV & YOLO

Build a real-time face recognition system using YOLO for face detection and OpenCV's LBPH recognizer for identifying known people from your own training photos.

## Imports

In [ ]:
# import the necessary libraries
import os
import cv2
import numpy as np
from ultralytics import YOLO
from PIL import Image
import json

## Collect images from the WebCam (Run only when you want to get new images)

- Run the following cell to capture the faces of peoples and save inside a folder with their names. 
- When you run the cell it automatically creates a folder inside the "faces/" directory, and captures a frame whenever you press the "s" key.
- Press "q" to quit/stop saving the images

In [2]:
from utils_vision import capture_faces

# Get and save images for a new person

#new_person = "Person1"
#capture_faces(person_name=new_person, camera_index=0)

## Data Pre-Processing

This section automatically discovers every person folder inside `faces`, assigns each folder a numeric label, and prepares the detected faces for training.

### Find Training Folders

Every subfolder inside `faces` is treated as a separate person. The folder names are loaded automatically, so users can name them whatever they want.

For example:

faces/
<br>
├── Mariya/
<br>
├── John/
<br>
└── Sarah/

will automatically produce the names `Mariya`, `John`, and `Sarah`.

In [3]:
# get the folders in the faces folder
faces_folder = "faces"

folders = [
    folder for folder in os.listdir(faces_folder)
    if os.path.isdir(os.path.join(faces_folder, folder))
]

# sort the folders in alphabetical order
folders.sort()
folders

['Aidah Beauty', 'Mama Nene', 'Mariya', 'Teacher']

### Detect and Prepare Faces

Each image is loaded from its person folder, YOLO detects the face, and the detected face is cropped, converted to grayscale, and resized to `200 × 200`.

The numeric labels are created automatically with `enumerate()`, while `label_names` keeps the mapping between each label and the actual folder name.

In [ ]:
# initialize the lists and dictionary
faces = []
labels = []
label_names = {}

# load the face model
face_model = YOLO("yolo_models/yolo26s-face.pt")

# loop through the folders and detect the face
for label, name in enumerate(folders):
    folder = os.path.join(faces_folder, name)
    
    
    label_names[label] = name # get the name of the person

    files = os.listdir(folder)

    # loop through the files in the folder and detect the face
    for file in files:
        photo_path = os.path.join(folder, file)
        photo = cv2.imread(photo_path)

        if photo is None:
            continue

        # detect the face
        face_result = face_model(photo, verbose=False)

        # if no face is detected, continue
        if len(face_result[0].boxes) == 0:
            continue

        # get the coordinates of the detected face
        left, top, right, bottom = face_result[0].boxes.xyxy[0].int()

        # crop the face, convert to grayscale, and resize to 200x200
        face = photo[top:bottom, left:right]
        face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
        face = cv2.resize(face, (200, 200))

        # append the face and label to the lists
        faces.append(face)
        labels.append(label)


# save the label_names dictionary as a json file to the faces folder 
with open("faces/label_names.json", "w") as f:
    json.dump(label_names, f)

# display an example face
Image.fromarray(faces[0])


## Training

Create an OpenCV LBPH face recognizer, train it using the processed faces and their numeric labels, then save the trained model to `face_recognizer.yml` so it can be loaded later for real-time recognition.

In [8]:
face_recognizer = cv2.face.LBPHFaceRecognizer_create()

face_recognizer.train(faces, np.array(labels))
face_recognizer.write("face_recognizer.yml")

## Inference

Run the trained face recognizer on a live webcam feed, display the predicted name and recognition distance, and label unfamiliar faces as `Unknown`.

> **Webcam note:** `cv2.VideoCapture(0)` uses camera device `0`, but your webcam index may be different. If the camera does not open, try `1`, `2`, or another available device number.

The default YOLO `face` label and confidence score are disabled here:

```python
processed_feed = face_result[0].plot(
    labels=False,
    conf=False
)
```

If you want to show YOLO's default face label and confidence again, comment out that block and uncomment:
```python
processed_feed = face_result[0].plot()
```

In [ ]:
import cv2
import json
from ultralytics import YOLO



cap = cv2.VideoCapture(0)

max_distance = 65 # max distance for the face recognizer

# load the face model
face_model = YOLO("yolo_models/yolo26s-face.pt") 
# load the label_names dictionary from the faces folder
with open("faces/label_names.json", "r") as f:
    label_names = json.load(f)

# load the trained model 
face_recognizer = cv2.face.LBPHFaceRecognizer_create()
face_recognizer.read("face_recognizer.yml")

# start the video capture and detect faces
while cv2.waitKey(1) != ord("q"):
    success, frame = cap.read()

    # Safety check to prevent NoneType errors
    if not success or frame is None:
        print("Failed to read from camera. Try changing the VideoCapture index.")
        break

    # flip the frame horizontally
    frame = cv2.flip(frame, 1)
    
    # detect the face
    face_result = face_model(frame, verbose=False) 
    
    # processed feed without the default face label
    processed_feed = face_result[0].plot(
        labels=False,
        conf=False
    )

    # loop through the detected faces and predict the label
    for box in face_result[0].boxes.xyxy:
        left, top, right, bottom = box.int()

        # crop the face, convert to grayscale, and resize to 200x200
        face = frame[top:bottom, left:right]
        face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
        face = cv2.resize(face, (200, 200))

        # predict the label and distance
        pred_label, distance = face_recognizer.predict(face)

        # if the distance is greater than the max distance, label as unknown
        if distance > max_distance:
            name = "Unknown"
        else:
            name = label_names[str(pred_label)]

        # draw the rectangle around the detected face
        label_height = 30
        blue = (255, 0, 0)

        cv2.rectangle(
            processed_feed,
            (int(left) - 2, int(bottom)),
            (int(right) + 2, int(bottom) + label_height),
            blue,
            -1
        )

        # add the name and distance to the processed feed
        cv2.putText(
            processed_feed,
            name + "  " + str(int(distance)),
            (int(left) + 2, int(bottom) + 20),
            0,
            0.6,
            (255, 255, 255),
            2
        )

    # display the processed feed
    cv2.imshow("my window", processed_feed)

# wait for 1 second and then release the camera and close all windows
cv2.waitKey(1000)
cap.release()
cv2.destroyAllWindows()

QFontDatabase: Cannot find font directory /home/usufcom/projects/ai_agent/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/usufcom/projects/ai_agent/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/usufcom/projects/ai_agent/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/usufcom/projects/ai_agent/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabas